# NB23: Redistricting Analysis

This notebook demonstrates the **redistricting data model** in siege_utilities:

- **RedistrictingPlan** — enacted/proposed/court-ordered plans per state/chamber/cycle
- **PlanDistrict** — individual districts with geometry and compactness scores
- **DistrictDemographics** — Census demographic overlays per district
- **PrecinctElectionResult** — precinct-level vote totals from RDH
- **PlanDistrictAssignment** — maps Seats to boundary polygons via GenericFK

We use the Pydantic schemas (no database required) to demonstrate the data model.

In [ ]:
from datetime import date

from siege_utilities.geo.schemas.redistricting import (
    RedistrictingPlanSchema,
    PlanDistrictSchema,
    DistrictDemographicsSchema,
    PrecinctElectionResultSchema,
)

## 1. RedistrictingPlan — Grouping Container

In [ ]:
# Virginia's 2020 enacted congressional plan
va_plan = RedistrictingPlanSchema(
    state_fips="51",
    chamber="congress",
    cycle_year=2020,
    plan_name="VA_congress_2021_enacted",
    plan_type="enacted",
    source="rdh",
    source_url="https://redistrictingdatahub.org/dataset/va-congress-2021/",
    num_districts=11,
    enacted_date=date(2021, 12, 28),
)
print(f"Plan: {va_plan.plan_name}")
print(f"State: {va_plan.state_fips}, Chamber: {va_plan.chamber}")
print(f"Cycle: {va_plan.cycle_year}, Districts: {va_plan.num_districts}")
print(f"Type: {va_plan.plan_type}, Source: {va_plan.source}")
print(f"Enacted: {va_plan.enacted_date}")

## 2. PlanDistrict — Districts with Compactness Scores

Each district stores compactness metrics that measure how "regularly shaped" it is.
Lower compactness may indicate gerrymandering.

In [ ]:
# Create districts with varying compactness
districts = []
for i in range(1, 12):
    d = PlanDistrictSchema(
        geoid=f"51{i:02d}",
        name=f"Virginia CD-{i}",
        vintage_year=2020,
        district_number=str(i),
        polsby_popper=round(0.15 + (i * 0.05), 2),  # Varies by district
        reock=round(0.20 + (i * 0.04), 2),
        total_population=770000 + (i * 5000),  # ~770K per district
        vap=600000 + (i * 3000),
        deviation_pct=round(-2.0 + (i * 0.4), 1),
    )
    districts.append(d)

print(f"{'District':<20} {'Polsby-Popper':>15} {'Reock':>10} {'Population':>12} {'Deviation':>10}")
print("-" * 70)
for d in districts:
    pp = d.polsby_popper or 0
    re = d.reock or 0
    print(f"{d.name:<20} {pp:>15.2f} {re:>10.2f} {d.total_population:>12,} {d.deviation_pct:>9.1f}%")

# Flag potentially gerrymandered districts
print("\nLow-compactness districts (Polsby-Popper < 0.30):")
for d in districts:
    if d.polsby_popper and d.polsby_popper < 0.30:
        print(f"  {d.name}: PP={d.polsby_popper:.2f}")

## 3. DistrictDemographics — Census Overlay

Each district gets a demographic profile from ACS/Decennial data.
The `values` JSON stores the full variable set; explicit columns enable fast queries.

In [ ]:
import pandas as pd

# Demographics for each district
demo_data = []
for d in districts:
    pop = d.total_population or 770000
    demo = DistrictDemographicsSchema(
        district_id=d.geoid,
        dataset="acs5",
        year=2022,
        pop_white=int(pop * 0.55),
        pop_black=int(pop * 0.20),
        pop_hispanic=int(pop * 0.12),
        pop_asian=int(pop * 0.08),
        pop_native=int(pop * 0.01),
        pop_other=int(pop * 0.04),
        median_household_income=55000 + int(d.geoid[-2:]) * 3000,
        poverty_rate=round(18.0 - int(d.geoid[-2:]) * 0.8, 1),
        values={"B01003_001E": pop, "B19013_001E": 55000 + int(d.geoid[-2:]) * 3000},
    )
    demo_data.append(demo)

# Display as DataFrame
df = pd.DataFrame([{
    "District": d.district_id,
    "White %": round(d.pop_white / sum(filter(None, [d.pop_white, d.pop_black, d.pop_hispanic, d.pop_asian, d.pop_native, d.pop_other])) * 100, 1) if d.pop_white else 0,
    "Black %": round(d.pop_black / sum(filter(None, [d.pop_white, d.pop_black, d.pop_hispanic, d.pop_asian, d.pop_native, d.pop_other])) * 100, 1) if d.pop_black else 0,
    "Hispanic %": round(d.pop_hispanic / sum(filter(None, [d.pop_white, d.pop_black, d.pop_hispanic, d.pop_asian, d.pop_native, d.pop_other])) * 100, 1) if d.pop_hispanic else 0,
    "Income": f"${d.median_household_income:,}",
    "Poverty": f"{d.poverty_rate}%",
} for d in demo_data])
print(df.to_string(index=False))

## 4. PrecinctElectionResult — Precinct-Level Votes

Precinct results from the Redistricting Data Hub (RDH) power
partisan analysis of redistricting plans.

In [ ]:
# Simulated precinct results
precincts = []
for p in range(1, 6):
    dem_votes = 400 + p * 50
    rep_votes = 350 + (6 - p) * 40
    total = dem_votes + rep_votes
    
    for party, votes, candidate in [
        ("DEM", dem_votes, "Smith"),
        ("REP", rep_votes, "Jones"),
    ]:
        precincts.append(PrecinctElectionResultSchema(
            state_fips="51",
            precinct_id=f"51-001-{p:04d}",
            election_year=2022,
            office="house",
            party=party,
            candidate_name=candidate,
            votes=votes,
            total_votes=total,
            vote_share=round(votes / total, 3),
        ))

# Summarize
print(f"{'Precinct':<18} {'Party':<6} {'Votes':>8} {'Share':>8}")
print("-" * 44)
for r in precincts:
    print(f"{r.precinct_id:<18} {r.party:<6} {r.votes:>8,} {r.vote_share:>7.1%}")

## 5. The Full Picture: Plan → District → Demographics → Votes

In production, these models form a hierarchy:

```
RedistrictingPlan (VA Congress 2020)
├── PlanDistrict (CD-1, geometry + compactness)
│   ├── DistrictDemographics (ACS 2022 overlay)
│   └── PlanDistrictAssignment → Seat (US_HOUSE VA-1)
├── PlanDistrict (CD-2, ...)
│   └── ...
└── PrecinctElectionResult (per-precinct votes)
```

The **PlanDistrictAssignment** uses a GenericFK to link any boundary model
(CongressionalDistrict, StateLegislativeUpper, etc.) to a Seat, enabling
"which boundary does this seat map to under this plan?" queries.

## Summary

| Model | Purpose | Key Fields |
|-------|---------|------------|
| **RedistrictingPlan** | Groups districts | state, chamber, cycle_year, plan_type |
| **PlanDistrict** | District geometry | polsby_popper, reock, total_population |
| **DistrictDemographics** | Census overlay | pop_white/black/hispanic, income, poverty |
| **PrecinctElectionResult** | Precinct votes | party, votes, vote_share |
| **PlanDistrictAssignment** | Seat → boundary | GenericFK to any boundary model |

**Compactness scores** (0-1 scale, higher = more compact):
- **Polsby-Popper**: area / perimeter² ratio
- **Reock**: area / minimum bounding circle ratio
- **Convex Hull**: area / convex hull area ratio
- **Schwartzberg**: perimeter / equal-area circle perimeter ratio